# ThinkingViT H1 — E1 · E3 · E4 실험 노트북 (의존성 보존 정리본)

**실행 순서:** 공통 준비 → E1/E3 공동 평가 코드 → E1 분석 → E3 분석 → E4 저장 가중치/평가 → 결과 보고서.

- E2 전용 프로파일링·벤치마크·탐색 셀은 제거했습니다. **공동 평가 스크립트 내부 E2 구현은 기존 E1/E3과 같은 평가·저장 구조를 공유하므로 보존했습니다.**
- 기존 결과 덮어쓰기를 막기 위해 비용이 큰 평가/재학습 및 추가 진단은 기본적으로 실행하지 않습니다 (`False`).
- 기존 Colab 출력·실행 횟수는 제거했습니다. **정적 의존성·문법 검사는 수행했지만, GPU/Drive 전체 실행 검증은 하지 않았습니다.**
- 원본 셀 번호는 코드 셀 메타데이터 `original_cell_index`에 기록했습니다.


## 1. 공통 환경 및 데이터 준비
순서대로 실행합니다.

In [ ]:
import os, subprocess, sys
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Colab → 런타임 변경 → GPU로 바꾸세요.'
ROOT=Path('/content/OnDevice')
if not (ROOT/'.git').exists():
    subprocess.run(['git','clone','-q','https://github.com/hee4040/OnDevice.git',str(ROOT)],check=True)
subprocess.run(['git','-C',str(ROOT),'fetch','-q','origin',
    'experiment/h1-diagnosis',
    'experiment/h1-attention-full-mlp-selective',
    'experiment/h1-selective-token-refinement'],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','-q','experiment/h1-diagnosis'],check=True)
# 필요한 두 구현만 다른 브랜치에서 가져오고, 사용자 원본 파일은 덮어쓰지 않습니다.
for branch,filename in [
  ('experiment/h1-attention-full-mlp-selective','mlp_selective_refinement.py'),
  ('experiment/h1-selective-token-refinement','selective_refinement.py')]:
    target=ROOT/'baseline'/filename
    source=subprocess.check_output(['git','-C',str(ROOT),'show',f'origin/{branch}:baseline/{filename}'])
    target.write_bytes(source)
print('Git HEAD:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
print('CUDA device:',torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
import subprocess
MINIFORGE=Path('/content/miniforge3')
PY=MINIFORGE/'envs/ondevice/bin/python'
if not (MINIFORGE/'bin/conda').is_file():
    subprocess.run(['wget','-q','https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh','-O','/tmp/miniforge.sh'],check=True)
    subprocess.run(['bash','/tmp/miniforge.sh','-b','-p',str(MINIFORGE)],check=True)
if not PY.is_file():
    subprocess.run([str(MINIFORGE/'bin/conda'),'create','-y','-n','ondevice','python=3.10','pip'],check=True)
repo=ROOT/'third_party/ThinkingViT'
if not (repo/'.git').is_dir():
    repo.parent.mkdir(parents=True,exist_ok=True)
    subprocess.run(['git','clone','-q','https://github.com/ds-kiel/ThinkingViT.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'checkout','-q','9254c4950d3159937ad682592d58a2be69708c42'],check=True)
# 설치 여부를 확인하고 누락한 경우에만 패키지를 설치합니다.
probe_env=os.environ.copy()
probe_env['PYTHONPATH']=str(repo)+os.pathsep+probe_env.get('PYTHONPATH','')
probe=subprocess.run([str(PY),'-c','import torch,torchvision,timm,scipy,matplotlib;print(torch.__version__)'],
    capture_output=True,text=True,env=probe_env)
if probe.returncode != 0:
    subprocess.run([str(PY),'-m','pip','install','-q','torch==2.5.1','torchvision==0.20.1',
                    '--index-url','https://download.pytorch.org/whl/cu124'],check=True)
    subprocess.run([str(PY),'-m','pip','install','-q','-r',str(repo/'requirements.txt'),
                    '-r',str(ROOT/'requirements-data.txt'),'scipy','matplotlib'],check=True)
print(subprocess.check_output([str(PY),'-c','import torch;print(torch.__version__, torch.cuda.is_available())'],text=True))
print('Official commit:',subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip())


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess
DRIVE=Path('/content/drive/MyDrive/ThinkingViT_E4')
archive=DRIVE/'imagenet1k_validation.tar.gz'
val=ROOT/'data/imagenet1k/validation'
assert archive.is_file(),f'Drive에서 validation 압축 백업을 찾지 못했습니다: {archive}'
def count_images(directory):
    return sum(p.suffix.lower() in {'.jpg','.jpeg','.png'} for p in directory.rglob('*') if p.is_file()) if directory.is_dir() else 0
if count_images(val)!=50000:
    (ROOT/'data/imagenet1k').mkdir(parents=True,exist_ok=True)
    subprocess.run(['tar','-xzf',str(archive),'-C',str(ROOT/'data/imagenet1k')],check=True)
classes=sum(p.is_dir() for p in val.iterdir())
count=count_images(val)
print('클래스:',classes,'이미지:',count)
assert classes==1000 and count==50000


In [ ]:
import hashlib,subprocess
ckpt=ROOT/'checkpoints/thinkingvit_3h6h_ema.pth'
if not ckpt.is_file():
    subprocess.run([str(PY),'scripts/prepare_weights.py'],cwd=str(ROOT),check=True)
assert ckpt.is_file()
digest=hashlib.sha256(ckpt.read_bytes()).hexdigest()
print('Checkpoint SHA256:',digest)
print('GitHub layerwise 예시 체크포인트 SHA256:',
      '9ead984e9e39887eb037a42c189c58c0af14cd9c2771ad27b4b01876997242b5')
print('일치 여부:',digest=='9ead984e9e39887eb037a42c189c58c0af14cd9c2771ad27b4b01876997242b5')
assert (ROOT/'baseline/evaluate_drop.py').is_file()
assert (ROOT/'baseline/evaluate_h1_diagnosis.py').is_file()


In [ ]:
from pathlib import Path
import torch

root = Path("/content/OnDevice")
py = Path("/content/miniforge3/envs/ondevice/bin/python")

checks = {
    "GPU 사용 가능": torch.cuda.is_available(),
    "OnDevice 프로젝트": (root / ".git").exists(),
    "ThinkingViT 원본": (root / "third_party/ThinkingViT/validate.py").exists(),
    "Conda Python": py.is_file(),
    "공식 가중치": (root / "checkpoints/thinkingvit_3h6h_ema.pth").is_file(),
    "ImageNet Validation": (root / "data/imagenet1k/validation").is_dir(),
    "E2 평가 스크립트": (root / "baseline/evaluate_h1_diagnosis.py").is_file(),
}

for name, ok in checks.items():
    print(f"{name}: {'정상' if ok else '없음'}")

print("전체 준비 완료:", all(checks.values()))

## 2. E1·E3 공동 평가 스크립트 생성
기존 `run_h1_e123.py` 평가 구현에 포함된 E2 부분은 E1/E3 출력 형식과 의존성이 연결되어 있어 유지합니다.

In [ ]:
%%writefile /content/OnDevice/baseline/run_h1_e123.py
#!/usr/bin/env python3
"""E1 full-attention/selective MLP; E2 fixed/adaptive full-k drop; E3 dense light update.

Requires OnDevice repo and ThinkingViT official custom timm. Full sample=50,000.
Output progress is resumable. E1 and E3 remain *dense accuracy oracles*, not sparse compute claims.
"""
import argparse
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
import time
import torch
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder

ROOT = Path('/content/OnDevice')
sys.path[:0] = [str(ROOT/'third_party/ThinkingViT'), str(ROOT/'baseline'), str(ROOT)]
from timm.data import create_transform, resolve_data_config
from evaluate_drop import load_model
from evaluate_h1_diagnosis import wide_input, wide_blocks
from selective_refinement import forward_narrow_with_attention, make_update_mask
from mlp_selective_refinement import forward_mlp_selective_wide

CONFIGS = ('3H_only', 'official_tau1', 'E1_mlp_75', 'E2_fixed_k5_drop50',
           'E2_adaptive_k2_8_drop50', 'E3_light_alpha025_50')


def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for buf in iter(lambda: f.read(1024*1024), b''):
            digest.update(buf)
    return digest.hexdigest()


def git_head(path):
    try:
        return subprocess.check_output(['git','-C',str(path),'rev-parse','HEAD'], text=True).strip()
    except (OSError, subprocess.CalledProcessError):
        return None


def atomic_save(state, path):
    temp = path.with_suffix('.partial.pt')
    torch.save(state, temp)
    os.replace(temp, path)


def light_update(model, x, p, update_mask, alpha):
    """Dense: all-token block computed, non-selected output blended towards original."""
    mask = update_mask.unsqueeze(-1)
    for block in model.blocks:
        candidate = block(x, p)
        x = torch.where(mask, candidate, x + alpha * (candidate - x))
    return x


def main():
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument('--samples', type=int, choices=(128,50000), required=True)
    ap.add_argument('--batch-size', type=int, default=8)
    ap.add_argument('--workers', type=int, default=2)
    ap.add_argument('--threshold', type=float, default=1.0)
    ap.add_argument('--split-entropy', type=float, default=2.0)
    ap.add_argument('--alpha', type=float, default=0.25)
    ap.add_argument('--seed', type=int, default=42)
    ap.add_argument('--save-every-batches', type=int, default=50)
    ap.add_argument('--output', type=Path, required=True)
    a = ap.parse_args()
    if not torch.cuda.is_available():
        raise RuntimeError('GPU 런타임이 필요합니다')
    if not (0 <= a.alpha <= 1):
        raise ValueError('alpha out of range')
    torch.manual_seed(a.seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    model = load_model(ROOT/'third_party/ThinkingViT', ROOT/'checkpoints/thinkingvit_3h6h_ema.pth')
    model.eval()
    val = ROOT/'data/imagenet1k/validation'
    ds = ImageFolder(val, transform=create_transform(**resolve_data_config({},model=model,use_test_size=True), is_training=False))
    assert len(ds) == 50000, f'validation count {len(ds)} != 50000'
    indices = torch.randperm(len(ds), generator=torch.Generator().manual_seed(a.seed))[:a.samples].tolist()
    a.output.mkdir(parents=True, exist_ok=True)
    progress = a.output/'progress.pt'
    args = {k:v for k,v in vars(a).items() if k!='output'}
    signature = {'args':args,'repo_commit':git_head(ROOT),'official_commit':git_head(ROOT/'third_party/ThinkingViT'),
                 'checkpoint_sha256':sha256(ROOT/'checkpoints/thinkingvit_3h6h_ema.pth'),
                 'script_sha256':sha256(Path(__file__)), 'imagenet_count':len(ds)}
    if (a.output/'summary.json').exists():
        old=json.loads((a.output/'summary.json').read_text())
        if old.get('signature') != signature:
            raise RuntimeError('Existing summary is from different settings; choose a new output folder')
        print('Already completed:', a.output/'summary.json')
        return
    if progress.exists():
        state=torch.load(progress,map_location='cpu',weights_only=False)
        if state['signature'] != signature:
            raise RuntimeError('Resume metadata changed; choose a new output folder')
        cursor=state['cursor']; correct=state['correct']; entropy=state['entropy']; route=state['route']; adaptive=state['adaptive']
        print('Resuming at',cursor,flush=True)
    else:
        cursor=0
        correct={name:torch.zeros(a.samples,dtype=torch.bool) for name in CONFIGS}
        entropy=torch.zeros(a.samples,dtype=torch.float32)
        route=torch.zeros(a.samples,dtype=torch.bool)
        adaptive={2:0,8:0}
    # A resumed run uses the exact same randperm index order, then skips finished entries.
    remaining=Subset(ds, indices[cursor:])
    loader=DataLoader(remaining,batch_size=a.batch_size,shuffle=False,num_workers=a.workers,pin_memory=True)
    started=time.time()
    with torch.inference_mode():
        for batch_number,(images,targets) in enumerate(loader,1):
            images=images.cuda(non_blocking=True); targets=targets.cuda(non_blocking=True)
            n=len(targets); end=cursor+n
            features, small, attention=forward_narrow_with_attention(model,images)
            e=model.entropy(small)[:,0]
            routed=e>=a.threshold
            ent_cpu=e.cpu(); entropy[cursor:end]=ent_cpu
            route[cursor:end]=routed.cpu()
            pred_small=small.argmax(1)
            correct['3H_only'][cursor:end]=(pred_small==targets).cpu()
            preds={name:pred_small.clone() for name in CONFIGS[1:]}
            idx=routed.nonzero(as_tuple=True)[0]
            if idx.numel():
                # True 6H stage width for official and all E1/E2/E3 conditions.
                p=model.p=model._stage_ratio(6)
                x=wide_input(model,images[idx],features[idx])
                a_selected=attention[idx]
                keep50=make_update_mask(a_selected,0.5,'topk')
                keep75=make_update_mask(a_selected,0.75,'topk')
                z=wide_blocks(model,x,p)
                preds['official_tau1'][idx]=model.forward_head(model.norm(z,int(p*model.embed_dim))).argmax(1)
                # E1: full attention on every token, mask only MLP residual (dense).
                z=forward_mlp_selective_wide(model,images[idx],features[idx],keep75)
                preds['E1_mlp_75'][idx]=model.forward_head(z).argmax(1)
                # E2: true sequence shorten after k full 6H blocks (k=5 vs adaptive k=2/8).
                z=wide_blocks(model,x,p,keep50,'drop',5)
                preds['E2_fixed_k5_drop50'][idx]=model.forward_head(model.norm(z,int(p*model.embed_dim))).argmax(1)
                hard=e[idx]>=a.split_entropy
                adaptive_pred=torch.empty(len(idx),device=images.device,dtype=torch.long)
                for use_hard,k in ((False,2),(True,8)):
                    sub=(hard==use_hard).nonzero(as_tuple=True)[0]
                    if sub.numel():
                        z=wide_blocks(model,x[sub],p,keep50[sub],'drop',k)
                        adaptive_pred[sub]=model.forward_head(model.norm(z,int(p*model.embed_dim))).argmax(1)
                        adaptive[k]+=len(sub)
                preds['E2_adaptive_k2_8_drop50'][idx]=adaptive_pred
                # E3: non-selected tokens receive fractional residual; dense block evaluation.
                z=light_update(model,x,p,keep50,a.alpha)
                preds['E3_light_alpha025_50'][idx]=model.forward_head(model.norm(z,int(p*model.embed_dim))).argmax(1)
            for name in CONFIGS[1:]:
                correct[name][cursor:end]=(preds[name]==targets).cpu()
            cursor=end
            if batch_number % a.save_every_batches==0 or cursor==a.samples:
                state={'signature':signature,'cursor':cursor,'correct':correct,'entropy':entropy,'route':route,'adaptive':adaptive}
                atomic_save(state,progress)
                print(f'Saved progress {cursor}/{a.samples}',flush=True)
    assert cursor==a.samples
    if a.samples==50000:
        official_acc=100*correct['official_tau1'].double().mean().item()
        assert abs(official_acc-80.304)<0.03, f'Official tau=1 mismatch {official_acc}; stop for audit'
        assert int(route.sum())==18007, f'Official routing mismatch {int(route.sum())}; stop for audit'
    counts=route.sum().item()
    rows=[]
    for name in CONFIGS:
        all_correct=correct[name]
        rows.append({'name':name,'samples':a.samples,'correct':int(all_correct.sum()),
                     'top1':100*all_correct.double().mean().item(),
                     'top1_routed':(100*all_correct[route].double().mean().item()) if counts else None,
                     'routed':int(counts),'compute_gmacs':None,
                     'latency_ms':None})
    summary={'experiment':'E1-E3','full_validation':a.samples==50000,
             'status':'complete' if a.samples==50000 else 'smoke_test_only',
             'samples':a.samples,'routed':int(counts),'adaptive_k_counts':adaptive,
             'accuracy_percent':{row['name']:row['top1'] for row in rows},'rows':rows,
             'signature':signature,'runtime_seconds_this_session':round(time.time()-started,2),
             'notes':['E1 and E3 are dense accuracy oracles: no measured compute benefit',
                      'E2 k=2/8 and k=5 have different computation; no cost-matched efficacy claim',
                      'No latency or GMACs measured in this script',
                      'Per-image correctness follows sorted ImageFolder + seed-42 randperm indices']}
    env={'gpu':torch.cuda.get_device_name(),'torch':torch.__version__,
         'cuda':torch.version.cuda,'python':platform.python_version(),
         'precision':'float32; TF32 allowed','signature':signature,
         'index_sha256':hashlib.sha256(torch.tensor(indices,dtype=torch.int32).numpy().tobytes()).hexdigest(),
         'dataset_root':str(val)}
    (a.output/'summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False)+'\n')
    (a.output/'environment.json').write_text(json.dumps(env,indent=2,ensure_ascii=False)+'\n')
    torch.save({'correct':correct,'entropy':entropy,'route':route,'indices':torch.tensor(indices,dtype=torch.int32)},a.output/'per_image.pt')
    import csv
    with (a.output/'summary.csv').open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=['name','samples','correct','top1','top1_routed','routed','compute_gmacs','latency_ms'])
        writer.writeheader();writer.writerows(rows)
    progress.unlink(missing_ok=True)
    print(json.dumps(summary,ensure_ascii=False,indent=2),flush=True)
    print('Saved:',a.output,flush=True)

if __name__=='__main__':main()


### 선택 실행: 128장 Smoke / 50,000장 재평가
이미 전체 결과가 저장된 경우 재평가를 실행하지 마세요.

In [ ]:
RUN_E123_SMOKE = False
if RUN_E123_SMOKE:
    import os,subprocess
    ENV=os.environ.copy()
    ENV['PYTHONPATH']=os.pathsep.join([str(ROOT/'third_party/ThinkingViT'),str(ROOT/'baseline'),str(ROOT),ENV.get('PYTHONPATH','')])
    subprocess.run([str(PY),'-u','baseline/run_h1_e123.py','--samples','128',
                    '--batch-size','8','--output',str(DRIVE/'h1_four_experiments/e123_smoke')],
                   check=True,cwd=str(ROOT),env=ENV)
else:
    print('128장 smoke test 비활성화')


In [ ]:
RUN_E123_FULL = False
if RUN_E123_FULL:
    import os,subprocess
    ENV=os.environ.copy()
    ENV['PYTHONPATH']=os.pathsep.join([str(ROOT/'third_party/ThinkingViT'),str(ROOT/'baseline'),str(ROOT),ENV.get('PYTHONPATH','')])
    subprocess.run([str(PY),'-u','baseline/run_h1_e123.py','--samples','50000',
                    '--batch-size','8','--workers','2','--threshold','1.0','--split-entropy','2.0',
                    '--alpha','0.25','--seed','42','--save-every-batches','50',
                    '--output',str(DRIVE/'h1_four_experiments/e123_full')],
                   check=True,cwd=str(ROOT),env=ENV)
else:
    print('50,000장 E1~E3 평가 비활성화. 기존 E4 데이터는 그대로 사용 가능합니다.')


## 3. E1 — Selective MLP (추가 진단 1,024장, 선택 실행)

In [ ]:
RUN_E1_RATIO_DIAG = False  # 선택 실행: E1 비율별 정확도
if RUN_E1_RATIO_DIAG:
    import sys
    import json
    import time
    import torch
    from pathlib import Path
    from torch.utils.data import DataLoader, Subset
    from torchvision.datasets import ImageFolder
    from timm.data import create_transform, resolve_data_config

    ROOT = Path("/content/OnDevice")
    DRIVE = Path("/content/drive/MyDrive/ThinkingViT_E4")
    OUT = DRIVE / "e1_ratio_diagnosis"
    OUT.mkdir(parents=True, exist_ok=True)

    sys.path[:0] = [
        str(ROOT / "third_party/ThinkingViT"),
        str(ROOT / "baseline"),
        str(ROOT),
    ]

    from evaluate_drop import load_model
    from evaluate_h1_diagnosis import wide_input, wide_blocks
    from selective_refinement import forward_narrow_with_attention
    from mlp_selective_refinement import (
        make_mlp_update_mask,
        forward_mlp_selective_wide,
    )

    # 기존 전체 평가 결과
    PER_IMAGE = DRIVE / "h1_four_experiments/e123_full/per_image.pt"

    assert PER_IMAGE.exists(), f"파일 없음: {PER_IMAGE}"

    saved = torch.load(
        PER_IMAGE,
        map_location="cpu",
        weights_only=True,
    )

    indices = saved["indices"][:1024].tolist()

    # 기존 50,000장 평가에서 사용한 정확한 데이터 경로
    VAL = Path("/content/OnDevice/data/imagenet1k/validation")

    assert VAL.is_dir(), f"ImageNet 경로가 없습니다: {VAL}"

    print("ImageNet 경로:", VAL)
    print("클래스 디렉터리 수:", sum(p.is_dir() for p in VAL.iterdir()))

    CKPT = ROOT / "checkpoints/thinkingvit_3h6h_ema.pth"
    REPO = ROOT / "third_party/ThinkingViT"

    model = load_model(REPO, CKPT, device="cuda")
    model.eval()

    config = resolve_data_config({}, model=model)
    transform = create_transform(
        **config,
        is_training=False,
    )

    dataset = ImageFolder(
        str(VAL),
        transform=transform,
    )

    assert max(indices) < len(dataset)

    loader = DataLoader(
        Subset(dataset, indices),
        batch_size=16,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )

    RATIOS = [0.75, 0.90, 0.95, 1.00]
    names = ["Official"] + [f"E1_{r:.2f}" for r in RATIOS]

    correct = {name: 0 for name in names}
    routed_correct = {name: 0 for name in names}
    routed_total = 0
    seen = 0

    # 기존 실험에서 사용한 라우팅 결과를 재사용
    route = saved["route"][:1024].bool()

    start = time.time()

    with torch.inference_mode():
        for images, targets in loader:
            batch_size = len(images)
            images = images.cuda(non_blocking=True)
            targets = targets.cuda(non_blocking=True)

            features, small_logits, attention = (
                forward_narrow_with_attention(model, images)
            )

            small_pred = small_logits.argmax(dim=1)

            batch_route = route[
                seen:seen + batch_size
            ].to(images.device)

            predictions = {
                name: small_pred.clone()
                for name in names
            }

            idx = batch_route.nonzero(as_tuple=True)[0]

            if idx.numel():
                selected_images = images[idx]
                selected_features = features[idx]
                selected_attention = attention[idx]

                p = model.p = model._stage_ratio(6)

                x = wide_input(
                    model,
                    selected_images,
                    selected_features,
                )

                z = wide_blocks(model, x, p)

                official_logits = model.forward_head(
                    model.norm(
                        z,
                        int(p * model.embed_dim),
                    )
                )

                predictions["Official"][idx] = (
                    official_logits.argmax(dim=1)
                )

                for ratio in RATIOS:
                    mask = make_mlp_update_mask(
                        selected_attention,
                        ratio,
                        "topk",
                    )

                    z_e1 = forward_mlp_selective_wide(
                        model,
                        selected_images,
                        selected_features,
                        mask,
                    )

                    logits_e1 = model.forward_head(z_e1)

                    name = f"E1_{ratio:.2f}"
                    predictions[name][idx] = (
                        logits_e1.argmax(dim=1)
                    )

                    if ratio == 1.0:
                        max_diff = (
                            official_logits - logits_e1
                        ).abs().max().item()

                        if max_diff > 1e-4:
                            raise RuntimeError(
                                f"100% Official 불일치: {max_diff}"
                            )

            for name in names:
                matches = predictions[name] == targets

                correct[name] += matches.sum().item()

                routed_correct[name] += (
                    matches[batch_route].sum().item()
                )

            routed_total += batch_route.sum().item()
            seen += batch_size

            if seen % 256 == 0:
                print(f"진행: {seen}/1024")

    results = {
        "samples": seen,
        "routed": routed_total,
        "elapsed_seconds": round(time.time() - start, 2),
        "accuracy": {},
    }

    print("\n===== E1 Ratio Accuracy Diagnosis =====")
    print("Samples:", seen)
    print("Routed :", routed_total)

    for name in names:
        accuracy = 100 * correct[name] / seen

        routed_accuracy = (
            100 * routed_correct[name] / routed_total
            if routed_total else None
        )

        results["accuracy"][name] = {
            "correct": correct[name],
            "top1_percent": accuracy,
            "routed_top1_percent": routed_accuracy,
        }

        print(
            f"{name:10s} "
            f"Top-1={accuracy:7.3f}% "
            f"Routed={routed_accuracy}"
        )

    output_path = OUT / "e1_ratio_accuracy_1024.json"

    output_path.write_text(
        json.dumps(results, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    print("\n저장 완료:", output_path)

## 4. E3 — Light Residual (추가 진단 1,024장, 선택 실행)

In [ ]:
RUN_E3_ALPHA_DIAG = False  # 선택 실행: E3 α별 정확도
if RUN_E3_ALPHA_DIAG:
    import sys
    import json
    import time
    from pathlib import Path

    import torch
    from torch.utils.data import DataLoader, Subset
    from torchvision.datasets import ImageFolder
    from timm.data import create_transform, resolve_data_config

    ROOT = Path("/content/OnDevice")
    REPO = ROOT / "third_party/ThinkingViT"
    DRIVE = Path("/content/drive/MyDrive/ThinkingViT_E4")

    sys.path[:0] = [
        str(REPO),
        str(ROOT / "baseline"),
        str(ROOT),
    ]

    from evaluate_drop import load_model
    from evaluate_h1_diagnosis import wide_input, wide_blocks
    from selective_refinement import (
        forward_narrow_with_attention,
        make_update_mask,
    )

    DEVICE = "cuda"
    CKPT = ROOT / "checkpoints/thinkingvit_3h6h_ema.pth"
    VAL = ROOT / "data/imagenet1k/validation"

    SAVED = DRIVE / "h1_four_experiments/e123_full/per_image.pt"
    OUT = DRIVE / "e3_alpha_diagnosis"
    OUT.mkdir(parents=True, exist_ok=True)

    N = 1024
    BATCH_SIZE = 8
    ALPHAS = [0.25, 0.50, 0.75, 1.00]

    assert CKPT.is_file()
    assert VAL.is_dir()
    assert SAVED.is_file()

    # 기존 평가의 이미지 순서 및 정답 기록
    previous = torch.load(
        SAVED,
        map_location="cpu",
        weights_only=True,
    )

    indices = previous["indices"][:N].long().tolist()
    old_route = previous["route"][:N]
    old_official = previous["correct"]["official_tau1"][:N]
    old_e3 = previous["correct"]["E3_light_alpha025_50"][:N]

    # 모델과 동일한 전처리
    model = load_model(REPO, CKPT, device=DEVICE)
    model.eval()

    ds = ImageFolder(
        VAL,
        transform=create_transform(
            **resolve_data_config(
                {},
                model=model,
                use_test_size=True,
            ),
            is_training=False,
        ),
    )

    assert len(ds) == 50000
    assert len(ds.classes) == 1000

    loader = DataLoader(
        Subset(ds, indices),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )

    names = ["Official"] + [f"E3_{a:.2f}" for a in ALPHAS]
    correct = {
        name: torch.zeros(N, dtype=torch.bool)
        for name in names
    }
    route_all = torch.zeros(N, dtype=torch.bool)

    start = time.time()
    cursor = 0

    with torch.inference_mode():
        for images, targets in loader:
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)

            b = len(targets)
            end = cursor + b

            # 3H 추론 및 라우팅
            features, small, attention = (
                forward_narrow_with_attention(model, images)
            )

            entropy = model.entropy(small)[:, 0]
            routed = entropy >= 1.0

            # 기존 50,000장 평가와 라우팅 일치 확인
            assert torch.equal(
                routed.cpu(), old_route[cursor:end]
            ), f"라우팅 불일치: {cursor}:{end}"

            route_all[cursor:end] = routed.cpu()

            # Early Exit 이미지는 모든 설정에서 동일한 3H 예측
            small_pred = small.argmax(dim=-1).reshape(-1)

            preds = {
                name: small_pred.clone()
                for name in names
            }

            idx = routed.nonzero(as_tuple=True)[0]

            if idx.numel() > 0:
                p = model.p = model._stage_ratio(6)

                x0 = wide_input(
                    model,
                    images[idx],
                    features[idx],
                )

                mask = make_update_mask(
                    attention[idx],
                    0.5,
                    "topk",
                )

                # Official
                z_official = wide_blocks(model, x0, p)

                official_logits = model.forward_head(
                    model.norm(
                        z_official,
                        int(p * model.embed_dim),
                    )
                )

                preds["Official"][idx] = (
                    official_logits.argmax(dim=-1)
                )

                # E3 alpha sweep
                for alpha in ALPHAS:
                    z = x0.clone()

                    for block in model.blocks:
                        candidate = block(z, p)

                        z = torch.where(
                            mask.unsqueeze(-1),
                            candidate,
                            z + alpha * (candidate - z),
                        )

                    logits = model.forward_head(
                        model.norm(
                            z,
                            int(p * model.embed_dim),
                        )
                    )

                    preds[f"E3_{alpha:.2f}"][idx] = (
                        logits.argmax(dim=-1)
                    )

                    # alpha=1은 Official과 수치적으로 동등해야 함
                    if alpha == 1.0:
                        assert torch.allclose(
                            logits,
                            official_logits,
                            atol=1e-4,
                            rtol=1e-4,
                        ), "E3 alpha=1.0 Logit 불일치"

            for name in names:
                correct[name][cursor:end] = (
                    preds[name] == targets
                ).cpu()

            cursor = end

            if cursor % 256 == 0 or cursor == N:
                print(f"진행: {cursor}/{N}")

    elapsed = time.time() - start

    # 기존 결과와 동일한지 검증
    official_match = torch.equal(
        correct["Official"],
        old_official,
    )
    e3_match = torch.equal(
        correct["E3_0.25"],
        old_e3,
    )

    print("\n===== 기존 평가 재현성 =====")
    print("Official 일치:", official_match)
    print("E3 alpha=0.25 일치:", e3_match)
    print("라우팅 이미지 수:", int(route_all.sum()))

    assert official_match, "Official 재현 실패"
    assert e3_match, "E3 alpha=0.25 재현 실패"

    # 정확도 집계
    routed_count = int(route_all.sum())

    results = {
        "samples": N,
        "routed": routed_count,
        "elapsed_seconds": elapsed,
        "accuracy": {},
    }

    print("\n===== E3 Alpha Accuracy Diagnosis =====")
    print(
        f"{'Configuration':<16}"
        f"{'Top-1':>12}"
        f"{'Routed Top-1':>17}"
        f"{'Delta (pp)':>14}"
    )

    official_acc = (
        correct["Official"].float().mean().item() * 100
    )

    for name in names:
        overall = correct[name].float().mean().item() * 100

        routed_acc = (
            correct[name][route_all].float().mean().item() * 100
        )

        delta = overall - official_acc

        results["accuracy"][name] = {
            "top1_percent": overall,
            "routed_top1_percent": routed_acc,
            "delta_vs_official_pp": delta,
        }

        print(
            f"{name:<16}"
            f"{overall:12.3f}"
            f"{routed_acc:17.3f}"
            f"{delta:14.3f}"
        )

    # 별도 저장: 기존 E1/E2/E3 결과는 변경하지 않음
    output_json = OUT / "e3_alpha_accuracy_1024.json"
    output_pt = OUT / "e3_alpha_per_image_1024.pt"

    output_json.write_text(
        json.dumps(
            results,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    torch.save(
        {
            "indices": torch.tensor(indices),
            "route": route_all,
            "correct": correct,
        },
        output_pt,
    )

    print("\n실행 시간:", round(elapsed, 1), "초")
    print("JSON 저장:", output_json)
    print("개별 이미지 결과:", output_pt)

## 5. E4 — Fine-tuning 체크포인트 및 평가 결과
기존 저장된 200 / 500 step 모델을 비교하며 기본값으로 새로 학습하지 않습니다.

In [ ]:
from pathlib import Path
import json
weights={
 'E4_200':DRIVE/'20261109T123318.734063Z-e4-parquet/student.pth',
 'E4_500':DRIVE/'e4_500steps/student.pth',
}
evals={
 'official':DRIVE/'eval_official_20261110',
 'E4_200':DRIVE/'eval_e4_200steps_20261110',
 'E4_500':DRIVE/'eval_e4_500steps_20261110',
}
for name,path in weights.items():
    print('WEIGHT',name,'OK' if path.is_file() else 'MISSING',path)
for name,path in evals.items():
    summary=path/'summary.json'
    pair=path/'per_image.pt'
    print('EVAL',name,'summary',summary.is_file(),'per_image',pair.is_file())
    if summary.is_file():
        d=json.loads(summary.read_text())
        print('  model',d.get('model'),'6H',d['top1'].get('wide_all'),
              'drop10',d['top1'].get('drop_10_all'),'drop25',d['top1'].get('drop_25_all'))


In [ ]:
RUN_MISSING_E4_EVALUATIONS = False
if RUN_MISSING_E4_EVALUATIONS:
    import os,subprocess,shutil,json
    ENV=os.environ.copy()
    ENV['PYTHONPATH']=os.pathsep.join([str(ROOT/'third_party/ThinkingViT'),str(ROOT/'baseline'),str(ROOT),ENV.get('PYTHONPATH','')])
    for tag in ('official','E4_200','E4_500'):
        destination=evals[tag]
        if (destination/'summary.json').is_file() and (destination/'per_image.pt').is_file():
            print('기존 백업 사용:',tag);continue
        cmd=[str(PY),'-u',str(ROOT/'baseline/evaluate_drop.py'),
             '--data-dir',str(ROOT/'data/imagenet1k'),
             '--checkpoint',str(ckpt),'--repo',str(ROOT/'third_party/ThinkingViT'),
             '--batch-size','8','--workers','2','--seed','42']
        if tag!='official':
            assert weights[tag].is_file(),f'학생 가중치 파일 없음: {weights[tag]}'
            cmd+=['--weights',str(weights[tag])]
        subprocess.run(cmd,cwd=str(ROOT),env=ENV,check=True)
        candidates=[]
        for sf in (ROOT/'results').glob('*-drop-eval-*/summary.json'):
            d=json.loads(sf.read_text())
            if ((tag=='official' and d.get('model')=='official') or
                (tag!='official' and d.get('weights')==str(weights[tag]))):
                candidates.append(sf.parent)
        assert candidates,f'{tag} 평가 생성 결과를 찾을 수 없음'
        src=max(candidates,key=lambda p:p.stat().st_mtime)
        assert (src/'per_image.pt').is_file()
        destination.mkdir(parents=True,exist_ok=True)
        for f in ('summary.json','per_image.pt'):
            shutil.copy2(src/f,destination/f)
        print('즉시 Drive 백업 완료:',tag,destination)
else:
    print('E4 재평가 비활성화. 기존 Drive 평가 요약 사용.')


In [ ]:
RUN_E4_TRAIN_FROM_200 = False  # 이미 완료: 바꾸지 마세요.
if RUN_E4_TRAIN_FROM_200:
    from huggingface_hub import get_token
    import subprocess,os
    script=ROOT/'baseline/finetune_curriculum_hybrid_parquet.py'
    resume=DRIVE/'20261109T123318.734063Z-e4-parquet/checkpoint.pt'
    assert script.is_file(),'원본 학습 스크립트 업로드 필요: baseline/finetune_curriculum_hybrid_parquet.py'
    assert resume.is_file(),'200 Step 체크포인트가 필요합니다.'
    assert get_token(),'학습용 Hugging Face 권한 로그인이 필요합니다.'
    env=os.environ.copy()
    env['HF_TOKEN']=get_token()
    env['PYTHONPATH']=os.pathsep.join([str(ROOT/'third_party/ThinkingViT'),str(ROOT/'baseline'),str(ROOT),env.get('PYTHONPATH','')])
    cmd=[str(PY),'-u',str(script),'--data-dir',str(ROOT/'data/imagenet1k'),
         '--checkpoint',str(ckpt),'--repo',str(ROOT/'third_party/ThinkingViT'),
         '--epochs','3.0','--batch-size','2','--lr','5e-5','--warmup','500',
         '--ce-weight','0.2','--seed','42','--max-steps','500','--log-every','10',
         '--save-every','50','--arrow-batch-size','128','--cache-dir','/content/imagenet_parquet_shards',
         '--resume',str(resume),'--output-dir',str(ROOT/'results/e4_resume_200_to_500')]
    subprocess.run(cmd,cwd=str(ROOT),env=env,check=True)
else:
    print('E4 학습 비활성화 — 기존 500 Step checkpoint/student.pth 보존')


### 선택 실행: E4 로그 및 가중치 분석

In [ ]:
RUN_E4_LOSS_ANALYSIS = False  # 선택 실행: E4 학습 로그
if RUN_E4_LOSS_ANALYSIS:
    from pathlib import Path
    import json
    import pandas as pd
    import matplotlib.pyplot as plt

    ROOT = Path("/content/drive/MyDrive/ThinkingViT_E4")

    runs = {
        "100": ROOT / "20261109T122620.464368Z-e4-parquet",
        "200": ROOT / "20261109T123318.734063Z-e4-parquet",
        "500": ROOT / "e4_500steps",
    }

    frames = []

    for name, folder in runs.items():
        path = folder / "train_log.jsonl"

        if not path.exists():
            print(f"{name}: 로그 없음")
            continue

        rows = []
        for line in path.read_text().splitlines():
            if line.strip():
                rows.append(json.loads(line))

        df = pd.DataFrame(rows)
        df["run"] = name
        frames.append(df)

        print(
            f"{name} Step 실험:",
            len(df), "개 로그",
            "| 기록 범위:",
            df["step"].min(), "~", df["step"].max()
        )

    if not frames:
        raise RuntimeError("학습 로그를 찾지 못했습니다.")

    all_logs = pd.concat(frames, ignore_index=True)
    all_logs = all_logs.sort_values(["step", "run"])

    # 동일 Step 중복 기록 확인
    duplicates = all_logs[
        all_logs.duplicated("step", keep=False)
    ]

    print("\n중복 Step 기록 수:", len(duplicates))

    # Step마다 하나의 기록만 사용
    logs = all_logs.drop_duplicates("step", keep="last")
    logs = logs.sort_values("step")

    print("\n===== 구간별 평균 손실 =====")

    logs["section"] = pd.cut(
        logs["step"],
        bins=[0, 100, 200, 300, 400, 500],
        labels=["1-100", "101-200", "201-300", "301-400", "401-500"]
    )

    columns = [
        "loss_3h",
        "loss_drop_6h",
        "loss_ce",
        "lr"
    ]

    print(
        logs.groupby("section", observed=True)[columns]
        .agg(["mean", "max"])
        .round(6)
        .to_string()
    )

    print("\n===== 손실 및 학습률 그래프 =====")

    fig, axes = plt.subplots(4, 1, figsize=(12, 13))

    for ax, col in zip(axes, columns):
        ax.plot(logs["step"], logs[col], marker=".", linewidth=1)
        ax.axvline(100, linestyle="--", alpha=0.5)
        ax.axvline(200, linestyle="--", alpha=0.5)
        ax.set_title(col)
        ax.set_xlabel("Training Step")
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("\n===== 마지막 로그 =====")
    print(logs.tail(10).to_string(index=False))

In [ ]:
RUN_E4_WEIGHT_ANALYSIS = False  # 선택 실행: E4 체크포인트 비교
if RUN_E4_WEIGHT_ANALYSIS:
    import torch
    import pandas as pd
    from pathlib import Path

    ROOT = Path("/content/drive/MyDrive/ThinkingViT_E4")

    paths = {
        100: ROOT / "20261109T122620.464368Z-e4-parquet/checkpoint.pt",
        200: ROOT / "20261109T123318.734063Z-e4-parquet/checkpoint.pt",
        500: ROOT / "e4_500steps/checkpoint.pt",
    }

    def load_weights(path):
        checkpoint = torch.load(
            path, map_location="cpu", weights_only=True
        )
        return {
            k: v.float().clone()
            for k, v in checkpoint["student"].items()
            if isinstance(v, torch.Tensor)
            and v.is_floating_point()
        }

    def compare_weights(a, b, label):
        rows = []

        for key in a.keys() & b.keys():
            x, y = a[key], b[key]

            if x.shape != y.shape:
                continue

            delta = torch.linalg.vector_norm(y - x).item()
            base = torch.linalg.vector_norm(x).item()

            rows.append({
                "parameter": key,
                "comparison": label,
                "relative_change_pct":
                    100 * delta / max(base, 1e-12),
                "absolute_change": delta,
            })

        return rows

    print("Loading checkpoints...")
    w100 = load_weights(paths[100])
    w200 = load_weights(paths[200])
    w500 = load_weights(paths[500])

    rows = (
        compare_weights(w100, w200, "100→200")
        + compare_weights(w200, w500, "200→500")
    )

    df = pd.DataFrame(rows)

    print("\n===== 변화량 상위 20개 =====")
    print(
        df.sort_values(
            "relative_change_pct", ascending=False
        ).head(20).to_string(index=False)
    )

    print("\n===== 구간별 변화량 요약 =====")
    print(
        df.groupby("comparison")["relative_change_pct"]
        .describe().round(4).to_string()
    )

    print("\n===== 가장 크게 변한 계층 =====")
    for comparison in ["100→200", "200→500"]:
        subset = df[df["comparison"] == comparison]
        print(f"\n{comparison}")
        print(
            subset.nlargest(15, "relative_change_pct")
            [["parameter", "relative_change_pct"]]
            .to_string(index=False)
        )

## 6. 보고서 내보내기

In [ ]:
%%writefile /content/OnDevice/baseline/export_h1_e1_e4_report.py
#!/usr/bin/env python3
"""Package H1 E1..E4 measurements in the repo's baseline/reports/DATE/h1-e1-e4 style.

Never fabricates missing experiments. Full ImageNet and per-image evaluation files remain
in Drive; only small summaries, optional plots, reproducibility notes are exported.
"""
import argparse
import csv
from datetime import datetime
import hashlib
import json
from pathlib import Path
import shutil

METRICS=[('3H 전체','narrow_all'),('6H 전체','wide_all'),('6H/3H τ=1','wide_threshold_1'),
         ('토큰 10%','drop_10_all'),('토큰 25%','drop_25_all'),('토큰 50%','drop_50_all'),
         ('토큰 75%','drop_75_all'),('토큰 90%','drop_90_all')]
DRIVE_PATHS={
'official':'eval_official_20261110',
'E4_200':'eval_e4_200steps_20261110',
'E4_500':'eval_e4_500steps_20261110',
}

def load_json(path):
    return json.loads(path.read_text(encoding='utf-8'))


def dump_json(path,obj):
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(obj,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')


def paired_analysis(a_path,b_path):
    if not (a_path.is_file() and b_path.is_file()):
        return None
    try:
        import torch
        from scipy.stats import binomtest
    except ImportError:
        return {'status':'skipped_missing_torch_or_scipy'}
    a=torch.load(a_path,map_location='cpu',weights_only=True)
    b=torch.load(b_path,map_location='cpu',weights_only=True)
    keys=['narrow_correct','wide_correct','drop_10_correct','drop_25_correct',
          'drop_50_correct','drop_75_correct','drop_90_correct']
    if not all(key in a and key in b for key in keys):
        return {'status':'skipped_missing_keys'}
    res={}
    for key in keys:
        aa=a[key].bool().flatten(); bb=b[key].bool().flatten()
        if aa.shape!=bb.shape or len(aa)!=50000:
            raise ValueError(f'Unmatched per-image length for {key}')
        improve=int((~aa & bb).sum()); worsen=int((aa & ~bb).sum()); n=len(aa)
        dif=bb.double()-aa.double()
        delta=100*float(dif.mean()); se=100*float(dif.std(unbiased=True))/(n**0.5)
        p=binomtest(improve,improve+worsen,p=.5).pvalue if improve+worsen else 1.
        res[key]={'improved':improve,'worsened':worsen,'delta_pp':delta,
                  'ci95_normal_pp':[delta-1.96*se,delta+1.96*se],
                  'mcnemar_exact_p':p,'bonferroni_7_p':min(1.,p*len(keys))}
    return {'status':'computed','note':'Assumes same sorted ImageFolder and seed-42 permutation; does not independently prove image identity',
            'n_tests':len(keys),'conditions':res}


def main():
    ap=argparse.ArgumentParser()
    ap.add_argument('--drive-base',type=Path,required=True)
    ap.add_argument('--e123-full',type=Path,required=True)
    ap.add_argument('--output',type=Path,required=True)
    args=ap.parse_args()
    out=args.output;out.mkdir(parents=True,exist_ok=True)
    (out/'accuracy').mkdir(exist_ok=True);(out/'e4').mkdir(exist_ok=True);(out/'figures').mkdir(exist_ok=True)
    e123_summary=args.e123_full/'summary.json'
    e123_present=e123_summary.is_file()
    full=False
    if e123_present:
        e123=load_json(e123_summary)
        full=(e123.get('samples')==50000 and e123.get('full_validation') is True
              and e123.get('status')=='complete')
        if full:
            for name in ('summary.json','summary.csv','environment.json'):
                p=args.e123_full/name
                if p.exists():shutil.copy2(p,out/'accuracy'/name)
    summaries={}
    for tag,rel in DRIVE_PATHS.items():
        path=args.drive_base/rel/'summary.json'
        if not path.exists():
            print('Missing E4 summary:',path)
            continue
        s=load_json(path)
        if 'top1' not in s: raise ValueError(f'Invalid summary: {path}')
        summaries[tag]=s
        shutil.copy2(path,out/'e4'/f'{tag}_summary.json')
    rows=[]
    for label,key in METRICS:
        row={'condition':label,'key':key}
        for tag in DRIVE_PATHS:
            value=summaries.get(tag,{}).get('top1',{}).get(key)
            row[tag]=round(float(value),5) if value is not None else None
        row['E4_200_minus_official_pp']=(round(row['E4_200']-row['official'],5) if row.get('E4_200') is not None and row.get('official') is not None else None)
        row['E4_500_minus_E4_200_pp']=(round(row['E4_500']-row['E4_200'],5) if row.get('E4_500') is not None and row.get('E4_200') is not None else None)
        rows.append(row)
    with (out/'e4'/'comparison.csv').open('w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
    dump_json(out/'e4'/'comparison.json',{'status':'completed_models_only','rows':rows,
        'interpretation':'A 500-step degradation is observed; logs alone do not identify a causal mechanism.'})
    tests={}
    for first,second in [('official','E4_200'),('E4_200','E4_500'),('official','E4_500')]:
        aa=args.drive_base/DRIVE_PATHS[first]/'per_image.pt'
        bb=args.drive_base/DRIVE_PATHS[second]/'per_image.pt'
        try:
            stat=paired_analysis(aa,bb)
        except Exception as exc:
            stat={'status':'error','message':str(exc)}
        if stat is not None: tests[first+'_vs_'+second]=stat
    if tests:dump_json(out/'e4'/'paired_statistics.json',tests)
    # E4 loss summary only covers resumed 210-500 checkpoint, not early steps.
    log=args.drive_base/'e4_500steps'/'train_log.jsonl'
    log_summary={'status':'not_found'}
    if log.exists():
        log_rows=[json.loads(line) for line in log.read_text().splitlines() if line.strip()]
        log_summary={'status':'found','log_steps':[min(x['step'] for x in log_rows),max(x['step'] for x in log_rows)],
                     'n_logged_batches':len(log_rows),'interval':'one logged batch every 10 training steps',
                     'not_mean_training_loss':True}
        for label,lo,hi in [('201_350',201,350),('351_500',351,500)]:
            part=[r for r in log_rows if lo<=r['step']<=hi]
            if part:
                log_summary[label]={'n_logged_batches':len(part),
                        'mean_logged_loss_3h':sum(r['loss_3h'] for r in part)/len(part),
                        'mean_logged_loss_drop_6h':sum(r['loss_drop_6h'] for r in part)/len(part),
                        'mean_logged_loss_ce':sum(r['loss_ce'] for r in part)/len(part),
                        'mean_lr':sum(r['lr'] for r in part)/len(part)}
        dump_json(out/'e4'/'training_log_summary.json',log_summary)
    # Only use available measurements; visual is not a new experiment.
    if all(tag in summaries for tag in DRIVE_PATHS):
        try:
            import matplotlib
            matplotlib.use('Agg')
            import matplotlib.pyplot as plt
            fig,ax=plt.subplots(figsize=(8.3,4.6))
            steps=[0,200,500]
            for key,label in [('wide_all','Full 6H'),('drop_10_all','Keep 10%'),('drop_25_all','Keep 25%')]:
                vals=[summaries[k]['top1'][key] for k in DRIVE_PATHS]
                ax.plot(steps,vals,marker='o',label=label)
            ax.set_xticks(steps,['Official','200 steps','500 steps'])
            ax.set_ylabel('Top-1 accuracy (%)');ax.set_title('Observed E4 evaluation results')
            ax.legend();ax.grid(alpha=.2)
            fig.tight_layout();fig.savefig(out/'figures'/'e4_accuracy.png',dpi=180)
            plt.close(fig)
        except ImportError:
            print('Matplotlib unavailable; skipping E4 plot.')
    details={
       'full_e123_evaluation_exported':full,
       'e123_status':'full 50,000-image evaluation' if full else 'missing or incomplete (not reported as result)',
       'e4_available':sorted(summaries),
       'e4_train_log':log_summary,
       'missing':[] if full else ['E1-E3 full summary/per_image; report fields intentionally left blank'],
       'notes':['No compute-matched E2 GMACs or E1/E3 measured sparse speedup',
                'No latency evaluation performed in this package',
                'Different experimental setups and model implementations must not be conflated',
                'Checkpoint state step=500 and scheduler last_epoch=500 reported from the saved checkpoint']}
    dump_json(out/'model-check.json',details)
    lines=[
      '# ThinkingViT H1 — E1~E4 실험 보고서 (근거 우선 초안)', '',
      '> 2026-11-10 기준. 첨부한 H1 보고서의 0~5장 서술 순서 및 Github `baseline/reports/` 구조를 참조함.', '',
      '## 한 줄 결론', '',
      'E4 200 Step에서 일부 token-drop 정확도 개선이 관찰됐지만, 500 Step에서는 주요 조건이 모두 하락했다. '
      'E1~E3은 전체 검증 결과가 확보되기 전에는 우열을 주장하지 않는다.', '',
      '## 0. 출발점', '',
      'ThinkingViT 3H→6H entropy routing(τ=1.0), ImageNet-1K validation 50,000장, '
      '3H CLS attention 기반 token importance를 기준으로 실험한다.', '',
      '## 1. 무엇을 바꾸었는가 / 실패 원인 가설', '',
      '- **E1**: full-token attention을 유지하고 비선택 패치의 MLP 잔차만 제외한다(75%; dense oracle).',
      '- **E2**: 6H 처음 k개의 block은 전체 토큰을 계산한 뒤 Top-K 50%를 남긴다. '
      '고정 k=5와 entropy 기반 k=2/8을 비교한다. 연산량이 다르므로 정확도만으로 우열을 결론 낼 수 없다.',
      '- **E3**: 비선택 토큰을 그대로 두지 않고 α=0.25 잔차 갱신을 적용한다(50%; dense oracle).',
      '- **E4**: curriculum+KD+CE 가중치 학생을 200/500 Step에서 평가한다.', '',
      '## 2. 기존 방법보다 나은가 (같은 연산량 비교)', '',
      f"E1~E3 전체 검증: **{'완료' if full else '미완료 — 아직 성능표 작성 불가'}**.",
      '동일 GMACs의 threshold 조정 또는 uniform drop 대조군을 계산/측정하기 전에는 '
      'E2의 비용 대비 우위를 주장하지 않는다. E1/E3은 현 구현이 dense이므로 이론적 추론 시간 단축을 주장하지 않는다.', '',
      '## 3. 파인튜닝 (E4)', '',
      '| 조건 | 공식 (%) | E4 200 (%) | E4 500 (%) | 500−200 (%p) |',
      '| --- | ---: | ---: | ---: | ---: |',
    ]
    for r in rows:
        def v(k): return '—' if r[k] is None else f'{r[k]:.3f}'
        lines.append(f"| {r['condition']} | {v('official')} | {v('E4_200')} | {v('E4_500')} | {v('E4_500_minus_E4_200_pp')} |")
    lines += ['', '500 Step 체크포인트: step=500, epoch=0, scheduler.last_epoch=500, lr≈5×10⁻⁵. '
                 '원본 200 Step에서 이어서 300 Step을 학습했다. 로그는 210~500 Step의 10 Step 간격 개별 배치만 기록했다.',
              '200→500 정확도 하락은 관찰 사실이지만 **학습률 상승이 원인이라는 인과 결론은 미확정**이다.', '',
              '![E4 관찰 정확도](figures/e4_accuracy.png)', '',
              '## 4. 실제로 빨라지나 (추론 시간)', '',
              '**미측정.** 이번 E1~E4 노트북은 정확도 검증 중심이며, GPU/CPU의 batch-1 pipeline latency를 측정하지 않는다. '
              '추후 정확도와 연산량이 검증된 후 별도 실측을 기록해야 한다.', '',
              '## 5. 한계와 다음 할 일', '',
              '- E1/E3는 dense oracle. E2와 비용 정합 비교가 필요함.',
              '- E4는 짧은 구간(최대 500 Step), 배치 크기 2이며 전체 3 epoch 훈련이 아님.',
              '- validation은 동일하게 이용되므로 선택/튜닝 결과에 독립 검증 편향 가능.',
              '- 200 Step 학습 구간 손실 로그는 현재 500 Step 백업 파일에 없음.',
              '- CPU/모바일 latency, GMACs, 다중 seed 재현은 별도 측정 필요.', '',
              '## 실행 결과 / 재현 정보', '',
              '- `accuracy/`: E1~E3 전체 검증 완료 시에만 생성되는 summary CSV·JSON, 환경',
              '- `e4/`: official·200·500 정확도와 대응표본 통계(원자료가 있는 경우)',
              '- `figures/`: 관측된 E4 결과 그래프',
              '- `model-check.json`: 미완료/한계/가용 파일을 명시',
              '- 큰 `.pt` 모델·개별 이미지 결과는 Google Drive에 두고 Git에는 추가하지 않음.', '',
              '기준 참조: https://github.com/hee4040/OnDevice/tree/codex/layerwise-drop-results', '']
    (out/'README.md').write_text('\n'.join(lines),encoding='utf-8')
    print('Exported:',out,'E1-E3 full=',full,'E4=',','.join(summaries))

if __name__=='__main__':main()


In [ ]:
RUN_EXPORT_REPORT = False  # 선택 실행: E1/E3/E4 보고서 재생성
if RUN_EXPORT_REPORT:
    import os
    import subprocess
    from pathlib import Path

    script = Path(
        "/content/OnDevice/baseline/export_h1_e1_e4_report.py"
    )

    assert script.is_file(), f"스크립트가 없습니다: {script}"

    cmd = [
        "/content/miniforge3/envs/ondevice/bin/python",
        "-u",
        str(script),
        "--drive-base",
        "/content/drive/MyDrive/ThinkingViT_E4",
        "--e123-full",
        "/content/drive/MyDrive/ThinkingViT_E4/h1_four_experiments/e123_full",
        "--output",
        "/content/drive/MyDrive/ThinkingViT_E4/h1_four_experiments/report/2026-11-10/h1-e1-e4",
    ]

    # Matplotlib 백엔드 충돌 해결
    env = os.environ.copy()
    env["MPLBACKEND"] = "Agg"

    print("===== 보고서 생성 재시도 =====", flush=True)

    result = subprocess.run(
        cmd,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    print("===== 표준 출력 =====")
    print(result.stdout or "(없음)")

    print("===== 오류 출력 =====")
    print(result.stderr or "(없음)")

    print("종료 코드:", result.returncode)

    if result.returncode == 0:
        print("보고서 생성 스크립트 실행 성공!")
    else:
        print("다른 오류가 발생했습니다.")

## 7. 해석 및 재현 시 주의

- **E1/E3은 dense 정확도 진단**이며 FLOPs 또는 추론시간 감소를 입증하지 않습니다.
- E4는 200/500-step 저장 결과를 비교합니다. 정확도 하락의 직접 원인은 검증되지 않았습니다.
- E2 전용 실행시간 벤치마크는 제외했으나 공동 평가 스크립트의 E2 계산은 보존했습니다.
- 이 실험은 팀에서 이미 수행한 방법의 재현·진단을 포함하며, 신규 독립적인 성능 향상으로 주장하지 않습니다.
